Azure provides three messaging and event services that serve distinct purposes:

| Service | AWS equivalent | Pattern | Use case |
|---|---|---|---|
| **Azure Service Bus** | Amazon SQS + SNS | Message broker | Enterprise messaging — ordered, transactional, reliable delivery |
| **Azure Event Grid** | Amazon EventBridge | Event router | Reactive event routing — lightweight notifications at low cost |
| **Azure Event Hubs** | Amazon Kinesis | Event streaming | High-throughput telemetry and log ingestion |

The key distinction:
- **Service Bus** — *messages* (data + instructions to act on); guarantees delivery, ordering, transactions
- **Event Grid** — *events* (small notifications that something happened); routes to many subscribers
- **Event Hubs** — *streams* (high-volume sequential data); retain and replay; analytics pipelines

## Azure Service Bus

**Azure Service Bus** is a fully managed enterprise message broker. It decouples senders and receivers and guarantees delivery, ordering, and exactly-once processing — making it the right choice for critical business workflows.

### Namespaces and tiers

All Service Bus entities (queues and topics) live in a **namespace**:

| Tier | Features | Use case |
|---|---|---|
| **Basic** | Queues only; 256 KB messages | Simple queuing, dev/test |
| **Standard** | Queues + topics; 256 KB messages | Standard messaging |
| **Premium** | Queues + topics; **100 MB** messages; VNet, private endpoint, geo-disaster recovery | Production, large messages, compliance |

### Queues

A **queue** provides point-to-point messaging — one sender, one receiver (competing consumers):

```
Producer → [Queue] → Consumer A
                   → Consumer B  (competing — only one receives each message)
```

| Property | Description |
|---|---|
| **Max message size** | 256 KB (Standard) / 100 MB (Premium) |
| **Max queue size** | 1 GB – 80 GB |
| **Message TTL** | Default 14 days; configurable |
| **Lock duration** | Time a consumer holds a message lock (default 60 s) |
| **Max delivery count** | After N failed deliveries, message goes to dead-letter queue |

### Topics and subscriptions

A **topic** provides publish-subscribe messaging — one sender, many receivers:

```
Producer → [Topic] → Subscription A → Consumer A
                   → Subscription B → Consumer B
                   → Subscription C → Consumer C
```

Each subscription receives its own copy of every message. Subscriptions can have **filters** to receive only a subset of messages:

| Filter type | Description |
|---|---|
| **Boolean filter** | TrueFilter (all messages) or FalseFilter (no messages) |
| **SQL filter** | SQL-like expression on message properties: `priority = 'high'` |
| **Correlation filter** | Match on specific message properties (faster than SQL filter) |

### Message settlement

Service Bus uses a **peek-lock** model — receiving a message locks it (makes it invisible) rather than deleting it:

| Action | Description |
|---|---|
| **Complete** | Processing succeeded — delete the message |
| **Abandon** | Processing failed — release lock so another consumer can retry |
| **Dead-letter** | Move to dead-letter queue (DLQ) — inspect and process manually |
| **Defer** | Park the message; retrieve later by sequence number |

### Dead-letter queue (DLQ)

Every queue and subscription has a built-in **dead-letter queue**. Messages land here when:
- Delivery count exceeds `MaxDeliveryCount`
- TTL expires
- Consumer explicitly dead-letters the message
- Filter evaluation fails

Monitor the DLQ in production — a growing DLQ indicates a processing problem.

### Advanced features

| Feature | Description |
|---|---|
| **Sessions** | Group related messages by `SessionId`; single consumer processes all messages in a session in order — for stateful workflows |
| **Scheduled messages** | Deliver a message at a future time |
| **Message deferral** | Park a message and retrieve by sequence number later |
| **Transactions** | Atomic operations across multiple entities in the same namespace |
| **Duplicate detection** | Discard duplicate messages within a configurable time window |
| **Auto-forward** | Automatically forward messages from one queue/subscription to another |
| **Geo-disaster recovery** | Replicate namespace metadata to a secondary namespace; failover in case of regional outage |

## Azure Event Grid

**Azure Event Grid** is a fully managed event routing service. It delivers lightweight event notifications from event *sources* to event *handlers* in near-real-time — with no polling, no persistent queue, and a pay-per-event pricing model.

```
Event Source  →  Event Grid Topic/System Topic  →  Subscription (filter)  →  Handler
```

### Event sources

| Type | Description |
|---|---|
| **Azure system topics** | Built-in events from Azure services — Blob Storage, Resource Groups, Event Hubs, Service Bus, etc. |
| **Custom topics** | Your application publishes events to a custom topic endpoint |
| **Partner topics** | Events from external SaaS partners (Salesforce, SAP, Auth0) |
| **Event Grid namespaces** | New model supporting MQTT and Pull delivery (IoT and edge scenarios) |

### Event schema

```json
{
  "id": "abc123",
  "source": "/subscriptions/.../storageAccounts/myaccount",
  "type": "Microsoft.Storage.BlobCreated",
  "time": "2025-01-01T10:00:00Z",
  "data": {
    "api": "PutBlob",
    "url": "https://myaccount.blob.core.windows.net/images/photo.jpg"
  }
}
```

### Event handlers

| Handler | Use case |
|---|---|
| **Azure Functions** | Serverless processing — most common |
| **Logic Apps** | Low-code workflow automation |
| **Event Hubs** | Buffer events for stream processing |
| **Service Bus queue/topic** | Reliable delivery with DLQ |
| **Storage Queue** | Simple durable delivery |
| **Webhook** | Any HTTPS endpoint |

### Event subscriptions and filtering

An **event subscription** connects a topic to a handler. You can filter events by:
- **Event type** — e.g. only `Microsoft.Storage.BlobCreated`
- **Subject prefix/suffix** — e.g. only blobs in `/blobServices/default/containers/images/`
- **Advanced filters** — filter on event data fields using operators (equals, contains, begins with)

### Delivery and retry

Event Grid retries delivery using an **exponential backoff** schedule for up to **24 hours** (configurable). After all retries fail, events can be forwarded to a **dead-letter storage container** (Blob Storage). Event Grid guarantees **at-least-once delivery**.

### Event Grid vs Service Bus

| | Event Grid | Service Bus |
|---|---|---|
| Message size | 1 MB max | 256 KB / 100 MB (Premium) |
| Ordering | No | Yes (sessions) |
| Transactions | No | Yes |
| Dead-letter | Blob Storage | Built-in DLQ |
| Retention | 24 hours (retry window) | Up to 14 days |
| Fan-out | Yes (many subscribers) | Topics with subscriptions |
| Use case | Reactive event notifications | Reliable business messages |

## Azure Event Hubs

**Azure Event Hubs** is a fully managed, real-time data streaming and event ingestion service capable of ingesting millions of events per second. It is the Azure equivalent of Apache Kafka — and natively supports the **Kafka protocol**, so existing Kafka producers and consumers can connect with minimal code changes.

### Core concepts

```
Producers  →  Event Hub (namespace/hub)  →  Partitions  →  Consumer Groups  →  Consumers
```

| Concept | Description |
|---|---|
| **Namespace** | Container for one or more Event Hubs; endpoint and access policies |
| **Event Hub** | A named stream — equivalent to a Kafka topic |
| **Partition** | Ordered, immutable sequence of events; unit of parallelism |
| **Consumer group** | Independent view of the stream — each group reads all events independently |
| **Offset** | Position of an event within a partition — consumers track their own offset |
| **Retention** | Events are kept for 1–7 days (Standard) or up to 90 days (Premium/Dedicated) |

### Partitions

Partitions are the key to Event Hubs' scalability:
- Events with the same **partition key** always go to the same partition — preserving order for that key
- Each partition can be read by one consumer per consumer group at a time
- More partitions = more parallelism; partition count is set at creation and **cannot be changed** (in most tiers)
- Recommended: set partition count to the expected maximum number of parallel consumers

### Throughput units / Processing units

| Tier | Capacity unit | Ingress | Egress |
|---|---|---|---|
| **Basic** | Throughput Unit (TU) | 1 MB/s or 1000 events/s per TU | 2 MB/s per TU |
| **Standard** | Throughput Unit (TU) | Same | Same; up to 20 TUs |
| **Premium** | Processing Unit (PU) | Higher; predictable | Higher; VNet, CMK |
| **Dedicated** | Capacity Unit (CU) | Exclusive cluster | Highest; 10 TB retention |

Standard tier supports **Auto-inflate** — automatically increases TUs when throughput is exceeded.

### Event Hubs Capture

**Capture** automatically archives the raw event stream to Azure Blob Storage or Azure Data Lake Storage Gen2 in **Avro** format — without any additional code:

```
Event Hubs  →  Capture  →  Blob Storage / ADLS Gen2
                            (Avro files, partitioned by time window)
```

Use Capture as the foundation for a data lake ingestion pipeline — downstream tools (Spark, Synapse, Databricks) process the Avro files.

### Schema Registry

Event Hubs includes a **Schema Registry** — a central repository for Avro and JSON schemas. Producers register a schema; consumers look up the schema by ID embedded in the event. This enforces schema governance and enables schema evolution.

### Kafka compatibility

Event Hubs exposes a Kafka-compatible endpoint on port **9093** (TLS). Change the broker address in your Kafka client config — no code changes:

```python
# Kafka producer config pointing to Event Hubs
config = {
    'bootstrap.servers': '<namespace>.servicebus.windows.net:9093',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanism': 'PLAIN',
    'sasl.username': '$ConnectionString',
    'sasl.password': '<connection-string>'
}
```

### Event Hubs vs Service Bus vs Event Grid

| | Event Hubs | Service Bus | Event Grid |
|---|---|---|---|
| Pattern | Stream | Message queue/broker | Event router |
| Throughput | Millions/sec | Thousands/sec | Millions/sec |
| Retention | Days (up to 90) | Up to 14 days | 24 hours (retry) |
| Ordering | Per partition | Per session | No |
| Replay | Yes (by offset) | No | No |
| Consumer groups | Yes (independent reads) | Competing consumers | Subscriptions |
| Use case | Telemetry, logs, analytics | Business workflows | Reactive events |

## Choosing the Right Service

```
Do you need ordered, reliable delivery with transactions?
  → Yes → Service Bus (queues for P2P, topics for pub/sub)

Do you need to react to events from Azure services or custom apps?
  → Yes → Event Grid

Do you need to ingest high-volume streams (telemetry, logs, clickstreams)?
  → Yes → Event Hubs

Do you need to replay events or run multiple independent consumers?
  → Yes → Event Hubs

Do you have an existing Kafka workload?
  → Yes → Event Hubs (Kafka-compatible endpoint)
```

**Common combinations:**
- **Event Hubs → Azure Functions** — serverless processing of each event batch
- **Event Hubs → Capture → Blob Storage → Synapse/Databricks** — real-time ingestion + batch analytics
- **Event Grid → Service Bus** — route Azure service events reliably with DLQ
- **Event Grid → Azure Functions** — serverless reaction to Blob Storage uploads, resource changes

## Working with Service Bus, Event Grid, and Event Hubs via Python

In [ ]:
# pip install azure-identity azure-servicebus azure-eventgrid azure-eventhub

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.servicebus import ServiceBusClient, ServiceBusMessage

credential = DefaultAzureCredential()
sb_namespace = "my-servicebus.servicebus.windows.net"
sb_client = ServiceBusClient(fully_qualified_namespace=sb_namespace, credential=credential)

In [ ]:
# Send messages to a queue
with sb_client.get_queue_sender("orders") as sender:
    # Single message
    msg = ServiceBusMessage(
        body='{"orderId": "ord-001", "amount": 49.99}',
        content_type="application/json",
        subject="NewOrder",
        session_id="customer-001"    # group messages by customer for ordered processing
    )
    sender.send_messages(msg)
    print("Sent order message")

    # Batch of messages (efficient — single network call)
    batch = sender.create_message_batch()
    for i in range(10):
        batch.add_message(ServiceBusMessage(f'{{"orderId": "ord-{i:03d}"}}'.encode()))
    sender.send_messages(batch)
    print("Sent batch of 10 messages")

In [ ]:
# Receive and process messages with peek-lock
with sb_client.get_queue_receiver("orders", max_wait_time=5) as receiver:
    for msg in receiver:
        try:
            body = str(msg)
            print(f"Processing: {body}")
            # ... do work ...
            receiver.complete_message(msg)   # success — delete from queue
        except Exception as e:
            print(f"Failed: {e}")
            receiver.abandon_message(msg)    # release lock — retry later

# Receive from dead-letter queue
with sb_client.get_queue_receiver(
    "orders",
    sub_queue="deadletter",
    max_wait_time=5
) as dlq_receiver:
    for msg in dlq_receiver:
        print(f"DLQ message: {str(msg)}  reason: {msg.dead_letter_reason}")
        dlq_receiver.complete_message(msg)

In [ ]:
# Send to a topic
with sb_client.get_topic_sender("notifications") as sender:
    msg = ServiceBusMessage(
        body='{"event": "order_shipped", "orderId": "ord-001"}',
        application_properties={"region": "us-east", "priority": "high"}
    )
    sender.send_messages(msg)
    print("Published to topic")

# Receive from a subscription
with sb_client.get_subscription_receiver(
    topic_name="notifications",
    subscription_name="email-service",
    max_wait_time=5
) as receiver:
    for msg in receiver:
        print(f"Email service received: {str(msg)}")
        receiver.complete_message(msg)

In [ ]:
from azure.eventgrid import EventGridPublisherClient, EventGridEvent
from azure.core.credentials import AzureKeyCredential
from datetime import datetime, timezone

# Publish events to a custom Event Grid topic
eg_endpoint = "https://my-topic.eastus-1.eventgrid.azure.net/api/events"
eg_key      = "<topic-access-key>"

eg_client = EventGridPublisherClient(eg_endpoint, AzureKeyCredential(eg_key))

events = [
    EventGridEvent(
        event_type="MyApp.Order.Placed",
        subject="orders/ord-001",
        data={"orderId": "ord-001", "customerId": "cust-042", "amount": 49.99},
        data_version="1.0"
    ),
    EventGridEvent(
        event_type="MyApp.Order.Shipped",
        subject="orders/ord-001",
        data={"orderId": "ord-001", "trackingId": "TRK-9999"},
        data_version="1.0"
    )
]
eg_client.send(events)
print(f"Published {len(events)} events to Event Grid")

In [ ]:
import json
from azure.eventhub import EventHubProducerClient, EventData

eh_namespace = "my-eventhubs.servicebus.windows.net"
eh_name      = "telemetry"

producer = EventHubProducerClient(
    fully_qualified_namespace=eh_namespace,
    eventhub_name=eh_name,
    credential=credential
)

with producer:
    # Create a batch (respects max batch size automatically)
    batch = producer.create_batch(partition_key="device-001")  # same partition for this device
    for i in range(50):
        event = {"deviceId": "device-001", "temperature": 22.5 + i * 0.1, "seq": i}
        batch.add(EventData(json.dumps(event)))
    producer.send_batch(batch)
    print("Sent 50 telemetry events for device-001")

In [ ]:
from azure.eventhub import EventHubConsumerClient
from azure.eventhub.extensions.checkpointstoreblobaio import BlobCheckpointStore

# Use Blob Storage for checkpointing (track offset per partition)
checkpoint_store = BlobCheckpointStore(
    blob_account_url="https://mystorageaccount.blob.core.windows.net",
    container_name="eventhub-checkpoints",
    credential=credential
)

consumer = EventHubConsumerClient(
    fully_qualified_namespace=eh_namespace,
    eventhub_name=eh_name,
    consumer_group="$Default",
    credential=credential,
    checkpoint_store=checkpoint_store
)

def on_event(partition_context, event):
    data = json.loads(event.body_as_str())
    print(f"Partition {partition_context.partition_id}: device={data['deviceId']} temp={data['temperature']}")
    partition_context.update_checkpoint(event)  # save offset to Blob Storage

# Start consuming (runs until stopped)
# with consumer:
#     consumer.receive(on_event=on_event, starting_position="-1")  # -1 = from beginning
print("Consumer configured — call consumer.receive() to start processing")

## Summary

| Concept | Key Takeaway |
|---|---|
| Service Bus queue | Point-to-point; competing consumers; at-least-once; peek-lock settlement |
| Service Bus topic | Publish-subscribe; each subscription gets its own copy; filter by SQL or correlation |
| Peek-lock | Message locked (invisible) during processing; complete/abandon/dead-letter/defer |
| Dead-letter queue | Messages that exceed MaxDeliveryCount or are explicitly rejected; built into every queue/subscription |
| Sessions | Group messages by SessionId for ordered, stateful processing; one consumer per session |
| Duplicate detection | Discard duplicate message IDs within a configurable window |
| Premium tier | 100 MB messages; VNet; private endpoint; geo-DR |
| Event Grid | Event router; near-real-time; at-least-once; 24-hour retry; pay-per-event |
| System topics | Built-in Azure service events (Blob created, resource deleted, etc.) |
| Custom topics | Your application publishes events to Event Grid |
| Event Grid dead-letter | Blob Storage container — events that fail all retries |
| Event Hubs | High-throughput streaming; millions of events/sec; Kafka-compatible |
| Partition | Ordered sequence; partition key routes events to same partition |
| Consumer group | Independent view of stream — multiple consumers can read all events independently |
| Offset / checkpoint | Position in partition; save to Blob Storage to resume after restart |
| Event Hubs Capture | Auto-archive to Blob/ADLS Gen2 in Avro format — no code needed |
| Auto-inflate | Standard tier automatically scales TUs when throughput limit is hit |
| SB vs EG vs EH | Service Bus = reliable business messages; Event Grid = reactive notifications; Event Hubs = high-volume streams |